# Comparação entre IA e pipeline matemático

Compara os métodos IA com a referência consolidada usando somente IDs comuns.
Mid resolution usa normal+RG dos sweeps atuais: treino 30, validação 270 e
teste 700. High resolution mantém os runs completos existentes.

O Dice geral inclui falhas registradas como zero. Cada método IA disponível
tem 200 exames nos quatro folds; a comparação não seleciona o melhor modelo
por exame. Os cenários condicionados aos óstios são apenas descritivos.


In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
for candidate in (NOTEBOOK_CWD, *NOTEBOOK_CWD.parents):
    src_dir = candidate / "src"
    if src_dir.exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break

from utils.project.runtime.notebook_env import (
    configure_notebook_environment,  # noqa: E402
)

REPO_ROOT = configure_notebook_environment()

### Funções de apresentação
Os helpers abaixo organizam os gráficos específicos desta análise.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from eda.comparisons.view.ia_vs_pipeline_views import (
    display_comparison_scenario,
    plot_ia_vs_math_scatter_interactive,
    plot_image_dice_scatter_interactive,
)

In [ ]:
from utils.comparison_utils.ia_math import (
    build_comparison_agg_df,
    filter_to_common_ia_math_ids,
    get_common_ia_math_keys,
    load_ia_results_for_comparison,
    load_math_results_for_comparison,
)
from utils.comparison_utils.ostia_scenarios import (
    build_ostia_image_comparison_df,
    load_ostia_comparison_scenario,
)
from utils.comparison_utils.paired_statistics import adjust_holm, compare_paired_dice
from utils.project.runtime.notebook_env import get_default_split_paths

sns.set_theme(
    style="whitegrid",
    context="notebook",
    palette="deep",
    rc={
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.dpi": 110,
        "savefig.dpi": 300,
    },
)
plt.close("all")

SPLIT_PATHS_BY_RESOLUTION = get_default_split_paths(REPO_ROOT)
VALID_SPLITS = ("train", "val", "test")

## Configuração

Ajuste nesta seção apenas os parâmetros da análise; o pipeline base não é alterado.

## Carregamento de dados

### Dados para comparação com métodos de IA

In [ ]:
IA_RESULTS_BASE = REPO_ROOT / "output/ia_results"
# A referência determinística vem somente de canonical, sem caminhos de sweep locais.
MATH_RESULT_SOURCES = {
    resolution: {
        split: numeric / f"results_{split}.csv"
        for split, numeric in split_paths.items()
    }
    for resolution, split_paths in SPLIT_PATHS_BY_RESOLUTION.items()
}

ia_results_df, missing_ia_files = load_ia_results_for_comparison(IA_RESULTS_BASE)
math_results_df, missing_math_files = load_math_results_for_comparison(
    MATH_RESULT_SOURCES
)
comparison_raw_df = pd.concat([ia_results_df, math_results_df], ignore_index=True)
comparison_common_df = filter_to_common_ia_math_ids(comparison_raw_df)
comparison_agg_df = build_comparison_agg_df(comparison_common_df)
common_ids_by_resolution = (
    get_common_ia_math_keys(comparison_raw_df)
    .groupby("target_resolution")["img_id"]
    .nunique()
)

print("Shapes carregados para comparação IA:")
print("- IA bruto:", ia_results_df.shape)
print("- Matemático bruto:", math_results_df.shape)
print("- Consolidado bruto:", comparison_raw_df.shape)
print("- Consolidado com IDs comuns:", comparison_common_df.shape)
print("- Agregado:", comparison_agg_df.shape)
print("\nIDs comuns IA x Matemático por resolução:")
print(common_ids_by_resolution)

if missing_ia_files:
    print("\nAvisos IA:")
    for msg in missing_ia_files:
        print("-", msg)

if missing_math_files:
    print("\nAvisos Matemático:")
    for msg in missing_math_files:
        print("-", msg)

## Análise

As subseções abaixo apresentam as métricas, tabelas ou visualizações do objetivo definido.

## Comparação: Métodos de IA vs Método Matemático

In [ ]:
# Cenários de óstios utilizam display_comparison_scenario importado de view.ia_vs_pipeline_views.

### 1. Conjunto completo de dados

In [ ]:
full_ia_df, full_math_df, full_missing_math_files = load_ostia_comparison_scenario(
    ia_results_df,
    MATH_RESULT_SOURCES,
    "full",
)
display_comparison_scenario(
    "Completo",
    full_math_df,
    full_ia_df,
    "mid_res",
)

In [ ]:
display_comparison_scenario(
    "Completo",
    full_math_df,
    full_ia_df,
    "high_res",
)

### 2. Apenas casos em que os óstios são encontrados corretamente

In [ ]:
correct_ia_df, correct_math_df, correct_missing_math_files = (
    load_ostia_comparison_scenario(
        ia_results_df,
        MATH_RESULT_SOURCES,
        "correct",
    )
)
display_comparison_scenario(
    "Apenas corretos",
    correct_math_df,
    correct_ia_df,
    "mid_res",
)

In [ ]:
display_comparison_scenario(
    "Apenas corretos",
    correct_math_df,
    correct_ia_df,
    "high_res",
)

### 3. Apenas casos em que os óstios não são encontrados corretamente

In [ ]:
incorrect_ia_df, incorrect_math_df, incorrect_missing_math_files = (
    load_ostia_comparison_scenario(
        ia_results_df,
        MATH_RESULT_SOURCES,
        "incorrect",
    )
)
display_comparison_scenario(
    "Apenas incorretos",
    incorrect_math_df,
    incorrect_ia_df,
    "mid_res",
)

In [ ]:
display_comparison_scenario(
    "Apenas incorretos",
    incorrect_math_df,
    incorrect_ia_df,
    "high_res",
)

## Comparação por imagem interativa: IA vs método matemático

Abaixo os gráficos interativos separados por cenário e resolução. Cada método de IA é apresentado como uma série independente, sem selecionar o melhor método por exame. Passe o mouse sobre os pontos para ver `img_id`, Dice e método.

### Completo

In [ ]:
# Completo - Mid Res
scenario_image_df = build_ostia_image_comparison_df(full_ia_df, full_math_df)
plot_image_dice_scatter_interactive(
    scenario_image_df, "mid_res", comparison_title="Completo"
)
plot_ia_vs_math_scatter_interactive(
    scenario_image_df, "mid_res", comparison_title="Completo"
)

In [ ]:
# Completo - High Res
scenario_image_df = build_ostia_image_comparison_df(full_ia_df, full_math_df)
plot_image_dice_scatter_interactive(
    scenario_image_df, "high_res", comparison_title="Completo"
)
plot_ia_vs_math_scatter_interactive(
    scenario_image_df, "high_res", comparison_title="Completo"
)

### Apenas corretos

In [ ]:
# Apenas corretos - Mid Res
scenario_image_df = build_ostia_image_comparison_df(correct_ia_df, correct_math_df)
plot_image_dice_scatter_interactive(
    scenario_image_df, "mid_res", comparison_title="Apenas corretos"
)
plot_ia_vs_math_scatter_interactive(
    scenario_image_df, "mid_res", comparison_title="Apenas corretos"
)

In [ ]:
# Apenas corretos - High Res
scenario_image_df = build_ostia_image_comparison_df(correct_ia_df, correct_math_df)
plot_image_dice_scatter_interactive(
    scenario_image_df, "high_res", comparison_title="Apenas corretos"
)
plot_ia_vs_math_scatter_interactive(
    scenario_image_df, "high_res", comparison_title="Apenas corretos"
)

### Apenas incorretos

In [ ]:
# Apenas incorretos - Mid Res
scenario_image_df = build_ostia_image_comparison_df(incorrect_ia_df, incorrect_math_df)
plot_image_dice_scatter_interactive(
    scenario_image_df, "mid_res", comparison_title="Apenas incorretos"
)
plot_ia_vs_math_scatter_interactive(
    scenario_image_df, "mid_res", comparison_title="Apenas incorretos"
)

In [ ]:
# Apenas incorretos - High Res
scenario_image_df = build_ostia_image_comparison_df(incorrect_ia_df, incorrect_math_df)
plot_image_dice_scatter_interactive(
    scenario_image_df, "high_res", comparison_title="Apenas incorretos"
)
plot_ia_vs_math_scatter_interactive(
    scenario_image_df, "high_res", comparison_title="Apenas incorretos"
)

## Wilcoxon pareado: pipeline versus IA

Cada comparação usa os mesmos IDs por método e resolução. A diferença é
pipeline menos IA; o efeito rank-biserial positivo favorece o pipeline.
Holm corrige a família das sete comparações disponíveis. O IC95% da diferença
média usa bootstrap pareado com 10.000 reamostragens e seed 42.
A inferência utiliza o cenário geral, sem excluir falhas dos óstios.


In [ ]:
# Verifica a cobertura antes de aplicar os testes nos dados pareados.
coverage_df = comparison_agg_df.loc[
    comparison_agg_df["source"].eq("ia"),
    [
        "target_resolution",
        "method",
        "n_samples",
        "n_folds_present",
        "expected_folds",
        "incomplete_coverage",
    ],
]
if coverage_df["incomplete_coverage"].any():
    raise ValueError("Há métodos IA sem todos os folds disponíveis.")
if len(coverage_df) != 7 or not coverage_df["n_samples"].eq(200).all():
    raise ValueError("Esperados sete métodos IA, com 200 exames cada.")

statistic_rows = []
for (resolution, method), ia_method_df in comparison_common_df.loc[
    comparison_common_df["source"].eq("ia")
].groupby(["target_resolution", "method"]):
    pipeline_df = comparison_common_df.loc[
        comparison_common_df["source"].eq("math")
        & comparison_common_df["target_resolution"].eq(resolution)
    ]
    statistic_rows.append(
        {
            "resolution": resolution,
            "method": method,
            **compare_paired_dice(
                ia_method_df, pipeline_df, id_column="img_id", dice_column="dice"
            ),
        }
    )
ia_paired_statistics = pd.DataFrame(statistic_rows)
ia_paired_statistics["p_holm"] = adjust_holm(ia_paired_statistics["p_value"])
display(
    ia_paired_statistics[
        [
            "resolution",
            "method",
            "paired_images",
            "baseline_mean_dice",
            "baseline_std_dice",
            "candidate_mean_dice",
            "candidate_std_dice",
            "mean_delta_dice",
            "wilcoxon_statistic",
            "p_value",
            "p_holm",
            "rank_biserial_effect",
            "mean_delta_ci_95_low",
            "mean_delta_ci_95_high",
        ]
    ]
)